In [21]:
# this script takes the file until 2025 Q2 and matches the columns song, album, country, platform


# Import main data file
import pandas as pd
import os
from opencc import OpenCC


file ='Earth_2022Q4_2025Q2_1_raw_combined_update.csv'

inputdirectory = '../../50 KM Group/Royalties/Statements/Karen/Earth/Combined Statements/'
outputdirectory = '../../50 KM Group/Royalties/Statements/Karen/_output/'
outputfilename1 = 'Earth_2022Q4_2025Q2_2_matched.csv'
outputfilename2 = 'Earth_2022Q4_2025Q2_2_matched_key_columns.csv'
globalinputdirectory = '../../50 KM Group/Royalties/Statements/Karen/All labels combined/lookup_tables/'

lookup_country = 'lookup_tables/Earth_Lookup_Country.csv'
#lookup_album = 'lookup_tables/Lookup_Album.csv'
#lookup_album_from_songs = 'lookup_tables/Lookup_Album_from_songs.csv'
lookup_platform = 'lookup_tables/Earth_Lookup_platform.csv'
#lookup_song = 'lookup_tables/Lookup_Song.csv'
#lookup_albumtype = 'lookup_tables/Lookup_album_type.csv'
lookup_isrc = 'lookup_tables/Earth_Lookup_isrc.csv'
lookup_song_album_albumtype = 'global_lookup_song_album_type.csv'

converter = OpenCC('s2t') 

def readfile(directory,file):
    path = os.path.join(directory, file)
    df = pd.read_csv(path,low_memory=False)
    print(f"The dataframe of the file '{file}' has {df.shape[0]} rows and {len(df.columns)} columns.")
    return df


data = readfile(inputdirectory,file)
print(f"Total fee: {data['Royalties (USD)'].sum()}. Total units: {data['Units'].sum()}")
#df_song = readfile(inputdirectory,lookup_song)
#df_album = readfile(inputdirectory,lookup_album)
df_country = readfile(inputdirectory,lookup_country)
df_platform = readfile(inputdirectory,lookup_platform)
#df_album_from_songs = readfile(inputdirectory,lookup_album_from_songs)
#df_albumtype = readfile(inputdirectory,lookup_albumtype)
df_isrc = readfile(inputdirectory,lookup_isrc)
df_song_album_albumtype = readfile(globalinputdirectory,lookup_song_album_albumtype)


The dataframe of the file 'Earth_2022Q4_2025Q2_1_raw_combined_update.csv' has 1271807 rows and 60 columns.
Total fee: 469133.40888972837. Total units: 290121307.0
The dataframe of the file 'lookup_tables/Earth_Lookup_Country.csv' has 404 rows and 2 columns.
The dataframe of the file 'lookup_tables/Earth_Lookup_platform.csv' has 93 rows and 2 columns.
The dataframe of the file 'lookup_tables/Earth_Lookup_isrc.csv' has 1516 rows and 2 columns.
The dataframe of the file 'global_lookup_song_album_type.csv' has 378 rows and 4 columns.


In [22]:

# Characterise the DAta frame
empty_rows_before_album_song = data[data['ISRC'].isna() &data['Album'].isna() & data['Song'].isna()]
count_of_rows = len(empty_rows_before_album_song)
print(f"\nThere are a total of {count_of_rows} Rows that have no entry in ISRC, 'Album' and 'Song'")
empty_rows_before_ISRC = data[data['ISRC'].isna()]
count_of_rows = len(empty_rows_before_ISRC)
print(f"\nThere are a total of {count_of_rows} Rows that have no entry in 'ISRC'")
data.fillna({'ISRC': 'XX_UNKNOWN'}, inplace=True)

# optimise_ISRC
#print("")
#print("Adding one column with harmonised ISRC values:")
#data = data.rename(columns={'ISRC': 'ISRC_original'})
#data['ISRC']=data['ISRC_original']
#data['ISRC']=data['ISRC'].str.replace('-','')
#print(f"The new dataframe has {data.shape[0]} rows and {len(data.columns)} columns.")

def merge(df1, df2, col):
    print(f"\nMerging on {col}:")
    empty_cells = df1[col].isna().sum()
    print(f"There are a total of {empty_cells} rows that have no entry in {col}")
    print(f"Total fee: {df1['Royalties (USD)'].sum()}. Total units: {df1['Units'].sum()}")
    df1.fillna({col: 'XX_UNKNOWN'}, inplace=True)
    df_merged = pd.merge(df1, df2, on=col, how='left')
    print(f"Total fee: {df_merged['Royalties (USD)'].sum()}. Total units: {df_merged['Units'].sum()}")
    new_columns = df_merged.columns.difference(df1.columns)
    first_new_col = new_columns[0] if not new_columns.empty else None
    empty_cells2 = df_merged[first_new_col].isna().sum() if first_new_col else 0
    diff_empty = empty_cells2 - empty_cells
    if diff_empty == 0:
        print(f"Merging of column {col} was successful")
    else:
        print(f"Merging with issues. There are a total of {diff_empty} cells that could not be matched (see 'match_issues_{col}.xlsx').")
        empty_rows = df_merged[df_merged[first_new_col].isna()].copy()
        empty_rows["Row_Number"] = empty_rows.index 
        col=col.replace('/','')
        empty_rows.to_excel(f"{outputdirectory}/match_issues_{col}.xlsx", engine='openpyxl', index=False)
    print(f"The new dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")
    return df_merged 

def mergealbum(df1, df2, col):
    print(f"\nMerging on {col}:")
    df_merged = pd.merge(df1, df2, on=col, how='left')
    new_columns = df_merged.columns.difference(df1.columns)
    first_new_col = new_columns[0] if not new_columns.empty else None
    empty_cells2 = df_merged[first_new_col].isna().sum() if first_new_col else 0
    if empty_cells2 == 0:
        print(f"Merging of column {col} was successful")
    else:
        print(f"Merging with issues. There are a total of {empty_cells2} cells that could not be matched (see 'match_issues_album_{col}.xlsx').")
        empty_rows = df_merged[df_merged[first_new_col].isna()]
        empty_rows.to_excel(f"{outputdirectory}/match_issues_album_{col}.xlsx", engine='openpyxl', index=False)
    print(f"The new dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")
    return df_merged 


df_merged = merge(data,df_country,'Country')
df_merged = merge(df_merged,df_platform,'Platform')


df_merged['Song'] = df_merged['Song'].astype(str)
#df_merged['Song_mod'] = (df_merged['Song']
#            .str.replace(" ", '', regex=False)
#            .str.replace("(", '', regex=False)
#            .str.replace(")", '', regex=False)
#            .str.replace("-", '', regex=False)
#            .str.replace("/", '', regex=False)
#            .str.replace('（', '', regex=False)
#            .str.replace('）', '', regex=False)
#            .str.replace('’', '', regex=False)
#            .str.replace("'", '', regex=False)
#            .str.lower())
#df_merged['Song_mod'] = df_merged['Song_mod'].apply(converter.convert)

#print("\nFixing: Album")
#df_merged.fillna({'Album': 'XX_UNKNOWN'}, inplace=True)
df_merged['Album'] = df_merged['Album'].astype(str)
#df_merged['Album_mod'] = (df_merged['Album']
#            .str.replace(" ", '', regex=False)
#            .str.replace("(", '', regex=False)
#            .str.replace(")", '', regex=False)
#            .str.replace("-", '', regex=False)
#            .str.replace("/", '', regex=False)
#            .str.replace('（', '', regex=False)
#            .str.replace('）', '', regex=False)
#            .str.replace('’', '', regex=False)
#            .str.replace("'", '', regex=False)
#            .str.lower())
#df_merged['Album_mod'] = df_merged['Album_mod'].apply(converter.convert)

#df_merged['Song_mod'] = df_merged['Song_mod'].apply(converter.convert)

df_merged['ISRC.Song.Album'] = df_merged['ISRC'] + '.' + df_merged['Song'] + '.' + df_merged['Album']
#df_merged['ISRC.Song.Album'] = (
#    df_merged[['ISRC_original', 'Song', 'Album']]
#    .fillna('')              # Replace NaN with empty string
#    .agg('.'.join, axis=1)   # Join with dots
#)
df_merged = merge(df_merged,df_isrc,'ISRC.Song.Album')
unused_rows = df_isrc[~df_isrc['ISRC.Song.Album'].isin(df_merged['ISRC.Song.Album'])]
output_path = os.path.join(outputdirectory, 'unused_lookup_rows_ISRC_Song_Album.csv')
unused_rows.to_csv(output_path, index=False)  

df_merged = merge(df_merged,df_song_album_albumtype,'ISRC (final)')
unused_rows = df_song_album_albumtype[~df_song_album_albumtype['ISRC (final)'].isin(df_merged['ISRC (final)'])]
output_path = os.path.join(outputdirectory, 'unused_lookup_rows_ISRC_final.csv')
unused_rows.to_csv(output_path, index=False)  

# Prepare for matching with album
#print("\nAdd copy of 2 columns for better matching:")
#df_merged['Song_copy']= df_merged['Song'].copy() 
#df_merged['Album_copy']= df_merged['Album'].copy() 
#df_merged.loc[:,'Song'] = df_merged['Song'].str.replace(' ', '', regex=False).str.lower()
#df_merged.loc[:,'Album'] = df_merged['Album'].str.replace(' ', '', regex=False).str.lower()
#print(f"The new dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")

# Merge with album: Split df_merged based on the condition whether there is an album entry or not
#print("\nSplitting daframe into those with album entry and those without:")
#df_merged_album = df_merged [df_merged['Album_mod'] != 'xx_unknown']
#print(f"The dataframe with album entries has {df_merged_album.shape[0]} rows and {len(df_merged_album.columns)} columns.")
#df_merged_no_album = df_merged [df_merged['Album_mod'] == 'xx_unknown']
#print(f"The dataframe without album entries has {df_merged_no_album.shape[0]} rows and {len(df_merged_no_album.columns)} columns.")

# Merge them each 
#df_merged_album = mergealbum(df_merged_album,df_album,'Album_mod')
#unused_rows = df_album[~df_album['Album_mod'].isin(df_merged_album['Album_mod'])]
#output_path = os.path.join(outputdirectory, 'unused_lookup_rows_album_mod.csv')
#unused_rows.to_csv(output_path, index=False)  
#print(f"Total fee: {df_merged_album['Royalties (USD)'].sum()}. Total units: {df_merged_album['Units'].sum()}")
#df_merged_album_from_songs = mergealbum(df_merged_no_album,df_album_from_songs,'Song_mod')
#unused_rows = df_album_from_songs[~df_album_from_songs['Song_mod'].isin(df_merged_album_from_songs['Song_mod'])]
#output_path = os.path.join(outputdirectory, 'unused_lookup_rows_album_from_songs_mod.csv')
#unused_rows.to_csv(output_path, index=False)  
#print(f"Total fee: {df_merged_album_from_songs['Royalties (USD)'].sum()}. Total units: {df_merged_album_from_songs['Units'].sum()}")

#empty_cells = df_merged_album_from_songs['Album new'].isna().sum()

# Concatenate the results
#print("")
#print("\nConcatenate the two:")
#df_merged = pd.concat([df_merged_album,df_merged_album_from_songs], ignore_index=True)
#print(f"The combined dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")

royalties = df_merged['Royalties (USD)'].sum()
units = df_merged['Units'].sum()
print(f"Total fee: {royalties}. Total units: {units}")
    
# merge with lookup_albumtype
#df_merged = merge(df_merged,df_albumtype,'Album new')
#unused_rows = df_albumtype[~df_albumtype['Album new'].isin(df_merged['Album new'])]
#output_path = os.path.join(outputdirectory, 'unused_lookup_rows_albumtype.csv')
#unused_rows.to_csv(output_path, index=False)  


# drop not needed 2 columns and rename others
print("\nDrop not needed columns:")
df_merged = df_merged.drop(columns=['ISRC.Song.Album'])
df_merged = df_merged.loc[:, ~df_merged.columns.str.startswith('Unnamed')]
#df_merged = df_merged.rename(columns={'Song': 'Song orig'})
#df_merged = df_merged.rename(columns={'Album': 'Album orig'})
#df_merged = df_merged.rename(columns={'Song new': 'Song'})
#df_merged = df_merged.rename(columns={'Album new': 'Album'})
df_merged = df_merged.rename(columns={'Platform': 'Platform orig'})
df_merged = df_merged.rename(columns={'Platform new': 'Platform'})
df_merged = df_merged.rename(columns={'Country': 'Country orig'})
df_merged = df_merged.rename(columns={'Country new': 'Country'})

df_merged.fillna({'Album (final)':'XX_UNKNOWN','Country':'XX_UNKNOWN','Platform':'XX_UNKNOWN','Song (final)':'XX_UNKNOWN','Album Type (final)':'XX_UNKNOWN'}, inplace=True)
df_merged[['Units','Royalties (USD)','Royalties (CNY)']]=df_merged[['Units','Royalties (USD)','Royalties (CNY)']].fillna(0)
print(f"The final dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")

def final_review(col):
    count = df_merged[col].str.contains('XX_UNKNOWN').sum()
    print(f"Rows with no entry for {col}: {count}")

df_merged = df_merged.sort_index(axis=1)

print(f"The final dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")


output_path = os.path.join(outputdirectory, outputfilename1)
df_merged.to_csv(output_path, index=False)

final_review('Country')
final_review('Platform')
final_review('Album (final)')
final_review('Song (final)')
final_review('Album Type (final)')

subset_df = df_merged[[
    'Statement Quarter',
    'Sales Quarter',
    'Sales Month',
    'Country',
    'Platform',
    'ISRC (final)',
    'Album (final)',
    'Units',
    'Share MABB (CNY)',
    'Royalties (CNY)'
]]

output_path = os.path.join(outputdirectory, outputfilename2)
subset_df.to_csv(output_path, index=False)








There are a total of 3 Rows that have no entry in ISRC, 'Album' and 'Song'

There are a total of 145 Rows that have no entry in 'ISRC'

Merging on Country:
There are a total of 711 rows that have no entry in Country
Total fee: 469133.40888972837. Total units: 290121307.0
Total fee: 469133.40888972837. Total units: 290121307.0
Merging with issues. There are a total of -711 cells that could not be matched (see 'match_issues_Country.xlsx').
The new dataframe has 1271807 rows and 61 columns.

Merging on Platform:
There are a total of 0 rows that have no entry in Platform
Total fee: 469133.40888972837. Total units: 290121307.0
Total fee: 469133.40888972837. Total units: 290121307.0
Merging of column Platform was successful
The new dataframe has 1271807 rows and 62 columns.

Merging on ISRC.Song.Album:
There are a total of 0 rows that have no entry in ISRC.Song.Album
Total fee: 469133.40888972837. Total units: 290121307.0
Total fee: 469133.40888972837. Total units: 290121307.0
Merging of co

In [23]:
for column in df_merged.columns:        
        print(column)

Album
Album (final)
Album Type (final)
Apple Identifier
Artist
Composer Name
Content
Copyright holder unique code
Country
Country orig
Currency
Device
FX Rate
Gross Amount
ISRC
ISRC (final)
Incentivized Ads - Monthly Share of Revenue
Incentivized Ads - Total Plays
Internal costs
Issuance
Noise Content
Non-incentivized Ads - Monthly Share of Revenue
Non-incentivized Ads - Total Downloads
Non-incentivized Ads - Total Plays
Paid/not paid
Period
Period end
Period start
Platform
Platform orig
Play Type
Price
Product
Product Type Identifier
Revenue
Royalties
Royalties (CNY)
Royalties (USD)
Royalty Rate
Sales Month
Sales Quarter
Sales Type
Sales or Return
Sales price
Settlement type
Share
Share Lyricist
Share MABB (CNY)
Share MABB (USD)
Share Master Owner
Share Performer
Share composer
Song
Song (final)
Song ID
Song Length
Statement Quarter
Streaming Subscription Category
Streaming Subscription Type
Streaming category
Streaming type
Total Downloads
UPC
Unit Price
Units
Withholding Tax


In [24]:
# for col in df_merged.columns:
#     print(f"Column: {col}")
#     print(df_merged[col].map(type).value_counts())
#     print()